# 03 — Plain baseline models (exploratory)

**Purpose.** Get a first, quick read on how four standard classifiers behave on the two feature sets (raw 14 features, extended 19 features) — and include **Gradient Boosting**, which the dataset creators' own paper reported as its best model.

**Reads:** `../data/processed/*` (written by notebook 02) · **Writes:** `../results/stage1_val_summary.csv`

**Evaluation design.** Models are trained on 80% of the training patients and scored on the remaining 20% (a validation part of the *training* set); the 2,417 test patients are never loaded, so the test set can be opened exactly once, in notebook 10. Every model and feature set gets identical treatment (same rows, same weights, same seed; one loop over a model dictionary), and per-class F1, the most-confused class pairs, a most-frequent-class dummy baseline and an independent recomputation of macro-F1 from the confusion matrix are reported.

**How to read the results (important).** These are single-split numbers on 1,934 validation rows; the smallest class (Reactive Arthritis) has only ~83 rows there, so its F1 is noisy. A single split can rank models wrongly. Therefore **nothing in this notebook selects a final model** — notebook 04 (fold-safe cross-validation) does that.

**Known small caveat.** The processed files were imputed and scaled using *all* training rows, including the ones we now hold out as validation. That gives a slight optimistic bias to every model equally. It is acceptable for an exploratory look; notebook 04 removes it by redoing preprocessing inside every fold.

## 0. Setup

**What this does:** imports libraries and prints their versions (results depend on them; compare with `results/environment.txt`), and makes sure the output folder exists.

In [1]:
import os, json, time, warnings
import numpy as np
import pandas as pd
import sklearn, xgboost

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score, classification_report, confusion_matrix
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier

print("pandas", pd.__version__, "| numpy", np.__version__, "| scikit-learn", sklearn.__version__, "| xgboost", xgboost.__version__)
os.makedirs('../results', exist_ok=True)

pandas 3.0.5 | numpy 2.5.2 | scikit-learn 1.9.0 | xgboost 3.4.1


## 1. Load the processed data and verify it

**What this does:** reads the six CSV files and the label mapping written by notebook 02, then checks them.

**Checks (all must pass):**
* shapes are exactly `(9668, 14)`/`(9668, 19)` for train and `(2417, …)` for test;
* features and labels line up row by row (same index, same order);
* there are no missing values left;
* the 7 class ids are 0–6 and the class names come from `label_mapping.json`.

**Reminder:** the test files (`X_*_test.csv`, `y_test.csv`) are deliberately NOT loaded in this notebook. The test set stays sealed until notebook 10.

In [2]:
P = '../data/processed/'
X_raw_train = pd.read_csv(P + 'X_raw_train.csv', index_col=0)
X_ext_train = pd.read_csv(P + 'X_ext_train.csv', index_col=0)
y_train     = pd.read_csv(P + 'y_train.csv',     index_col=0).iloc[:, 0]
with open(P + 'label_mapping.json', encoding='utf-8') as f:
    label_mapping = json.load(f)
disease_names = [k for k, v in sorted(label_mapping.items(), key=lambda kv: kv[1])]

assert X_raw_train.shape == (9668, 14) and X_ext_train.shape == (9668, 19)
assert X_raw_train.index.equals(y_train.index) and X_ext_train.index.equals(y_train.index)
assert X_raw_train.isna().sum().sum() == 0 and X_ext_train.isna().sum().sum() == 0
assert sorted(y_train.unique()) == list(range(7))
print("Loaded and verified. Classes:", dict(enumerate(disease_names)))
print("\nraw features:", list(X_raw_train.columns))
print("\nextra columns in ext:", [c for c in X_ext_train.columns if c not in X_raw_train.columns])

Loaded and verified. Classes: {0: 'Ankylosing Spondylitis', 1: 'Normal', 2: 'Psoriatic Arthritis', 3: 'Reactive Arthritis', 4: 'Rheumatoid Arthritis', 5: "Sjögren's Syndrome", 6: 'Systemic Lupus Erythematosus'}

raw features: ['Age', 'Gender', 'ESR', 'CRP', 'RF', 'Anti-CCP', 'HLA-B27', 'ANA', 'Anti-Ro', 'Anti-La', 'Anti-dsDNA', 'Anti-Sm', 'C3', 'C4']

extra columns in ext: ['RF_was_missing', 'Anti-CCP_was_missing', 'Inflammation_Score', 'C3_C4_Ratio', 'Autoantibody_Count']


## 2. Carve a validation set out of the training data

**What this does:** splits the 9,668 training rows 80/20 into **fit** (7,734) and **validation** (1,934) parts, stratified by disease so class proportions are preserved. The same rows are used for the raw and the extended feature set, so the two are directly comparable.

**Why:** models are trained on the fit part and scored on the validation part, so the real test set stays sealed.

**Check:** the class proportions of the two parts should match to within about 0.5 percentage points.

In [3]:
fit_idx, val_idx = train_test_split(
    y_train.index, test_size=0.2, stratify=y_train, random_state=42
)
y_fit, y_val = y_train.loc[fit_idx], y_train.loc[val_idx]
data = {
    'raw': (X_raw_train.loc[fit_idx], X_raw_train.loc[val_idx]),
    'ext': (X_ext_train.loc[fit_idx], X_ext_train.loc[val_idx]),
}
print("fit:", len(fit_idx), "| validation:", len(val_idx))
prop = pd.DataFrame({'fit': y_fit.value_counts(normalize=True),
                     'val': y_val.value_counts(normalize=True)}).sort_index().round(3)
prop.index = disease_names
prop['val_rows'] = y_val.value_counts().sort_index().values
print(prop)
assert (prop['fit'] - prop['val']).abs().max() < 0.005

fit: 7734 | validation: 1934
                                fit    val  val_rows
Ankylosing Spondylitis        0.176  0.176       340
Normal                        0.133  0.133       257
Psoriatic Arthritis           0.148  0.147       285
Reactive Arthritis            0.043  0.043        83
Rheumatoid Arthritis          0.236  0.236       456
Sjögren's Syndrome            0.153  0.153       296
Systemic Lupus Erythematosus  0.112  0.112       217


## 3. Evaluation helper

**What this does:** defines one function that turns a set of predictions into the metrics we track.

* **Macro-F1** — the average of the 7 per-class F1 scores; every class counts equally, so weak minority-class performance cannot hide behind the big classes. This is our main metric.
* **Weighted-F1** — per-class F1 weighted by class size (closer to overall accuracy). Reported for comparison with other papers.
* **Accuracy** — reported because most papers on this dataset quote it.
* **Per-class F1** — kept as a vector so we can see *which* diseases each model struggles with.

In [4]:
def evaluate(y_true, y_pred):
    return {
        'macro_f1':    f1_score(y_true, y_pred, average='macro'),
        'weighted_f1': f1_score(y_true, y_pred, average='weighted'),
        'accuracy':    accuracy_score(y_true, y_pred),
        'per_class_f1': f1_score(y_true, y_pred, average=None, labels=list(range(7))),
    }

## 4. Floor: the "always predict the biggest class" baseline

**What this does:** a dummy model that ignores all features and always answers "Rheumatoid Arthritis" (the most frequent class).

**Why:** any real model has to beat this by a large margin, otherwise it has learned nothing. It also shows why plain accuracy is misleading here: the dummy scores ~24% accuracy but its macro-F1 is close to 0.05.

**Expected:** accuracy ≈ 0.236, macro-F1 ≈ 0.05.

In [5]:
dummy = DummyClassifier(strategy='most_frequent').fit(data['ext'][0], y_fit)
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    dummy_res = evaluate(y_val, dummy.predict(data['ext'][1]))
print(f"Dummy (most frequent): accuracy={dummy_res['accuracy']:.3f}  macro-F1={dummy_res['macro_f1']:.3f}")

Dummy (most frequent): accuracy=0.236  macro-F1=0.055


## 5. Define the four models

**What this does:** describes each model with fixed settings used throughout the project (no tuning yet — tuning is notebook 07, with nested CV).

| Model | Why it is here | Class-imbalance handling |
|---|---|---|
| **Logistic Regression** (lbfgs, multinomial, `max_iter=1000`) | Linear reference; shows how much is non-linear structure | `class_weight='balanced'` |
| **Random Forest** (300 trees, unlimited depth) | Bagged trees; the most stable model in cross-validation | `class_weight='balanced'` |
| **XGBoost** (300 rounds, depth 5, lr 0.1) | Regularised gradient boosting | per-row `sample_weight` (balanced) |
| **Gradient Boosting** (sklearn; 300 rounds, depth 3, lr 0.1) | Plain gradient boosting, the winner in the dataset creators' paper | per-row `sample_weight` (balanced), because sklearn's `GradientBoostingClassifier` has no `class_weight` argument |

**What "balanced" means:** each row gets a weight inversely proportional to its class frequency, so an error on Reactive Arthritis (4.3% of data) costs about 5.5× more than an error on Rheumatoid Arthritis. Giving all four models the same balancing keeps the comparison fair. Whether balancing is the *best* way to handle imbalance is tested properly in notebook 05 (SMOTE vs class weights).

**Speed note:** Gradient Boosting builds 7 trees per round (one per class) and cannot use multiple cores, so expect roughly 1–3 minutes for each of its two runs.

In [6]:
models = {
    'Logistic Regression': lambda: LogisticRegression(solver='lbfgs', class_weight='balanced',
                                                      max_iter=1000, random_state=42),
    'Random Forest':       lambda: RandomForestClassifier(n_estimators=300, class_weight='balanced',
                                                          random_state=42, n_jobs=-1),
    'XGBoost':             lambda: XGBClassifier(n_estimators=300, max_depth=5, learning_rate=0.1,
                                                 objective='multi:softprob', eval_metric='mlogloss',
                                                 random_state=42, n_jobs=-1),
    'Gradient Boosting':   lambda: GradientBoostingClassifier(n_estimators=300, learning_rate=0.1,
                                                              max_depth=3, random_state=42),
}
needs_sample_weight = {'XGBoost', 'Gradient Boosting'}
sw_fit = compute_sample_weight(class_weight='balanced', y=y_fit)

print("models:", list(models))
print("sample-weight range (min..max):", sw_fit.min().round(3), "..", sw_fit.max().round(3))

models: ['Logistic Regression', 'Random Forest', 'XGBoost', 'Gradient Boosting']
sample-weight range (min..max): 0.606 .. 3.348


## 6. Train and score all 8 configurations (4 models × 2 feature sets)

**What this does:** for each model and each feature set: fit on the fit part, predict the validation part, record the metrics and the fit time. The predictions are kept so we can analyse errors afterwards.

**Expected runtime:** about 4–8 minutes in total, mostly Gradient Boosting.

**What to look at:** the printed line for each run. Tree-based models should clearly beat Logistic Regression; if a tree model scores below ~0.75 macro-F1, something is wrong.

In [7]:
rows, per_class, preds = [], {}, {}
for mname, make in models.items():
    for fname in ('raw', 'ext'):
        Xf, Xv = data[fname]
        model = make()
        t0 = time.time()
        if mname in needs_sample_weight:
            model.fit(Xf, y_fit, sample_weight=sw_fit)
        else:
            model.fit(Xf, y_fit)
        secs = time.time() - t0
        y_hat = model.predict(Xv)
        res = evaluate(y_val, y_hat)
        key = f"{mname} ({fname})"
        rows.append({'model': mname, 'features': fname, 'n_features': Xf.shape[1],
                     'macro_f1': res['macro_f1'], 'weighted_f1': res['weighted_f1'],
                     'accuracy': res['accuracy'], 'fit_seconds': secs})
        per_class[key] = res['per_class_f1']
        preds[key] = y_hat
        print(f"{key:<32} macro-F1={res['macro_f1']:.4f}  weighted-F1={res['weighted_f1']:.4f}  "
              f"acc={res['accuracy']:.4f}  ({secs:.0f}s)")

Logistic Regression (raw)        macro-F1=0.7781  weighted-F1=0.7923  acc=0.7885  (0s)
Logistic Regression (ext)        macro-F1=0.7842  weighted-F1=0.7991  acc=0.7947  (0s)
Random Forest (raw)              macro-F1=0.7982  weighted-F1=0.8147  acc=0.8170  (1s)
Random Forest (ext)              macro-F1=0.8074  weighted-F1=0.8217  acc=0.8226  (1s)
XGBoost (raw)                    macro-F1=0.7847  weighted-F1=0.7968  acc=0.7963  (2s)
XGBoost (ext)                    macro-F1=0.8048  weighted-F1=0.8187  acc=0.8185  (2s)
Gradient Boosting (raw)          macro-F1=0.8019  weighted-F1=0.8159  acc=0.8154  (26s)
Gradient Boosting (ext)          macro-F1=0.8044  weighted-F1=0.8144  acc=0.8144  (39s)


## 7. Summary table

**What this does:** collects the 8 results (plus the dummy floor) in one table, sorted by macro-F1.

**How to read it:** look at the *gaps*, not the ranking. If the top models are within ~0.01 of each other, they are statistically indistinguishable on a single 1,934-row split — that is exactly what cross-validation in notebook 04 resolves. Also compare `raw` vs `ext` for each model: earlier work found the extended features gave little or no gain.

In [8]:
summary = pd.DataFrame(rows).sort_values('macro_f1', ascending=False).reset_index(drop=True)
summary[['macro_f1', 'weighted_f1', 'accuracy']] = summary[['macro_f1', 'weighted_f1', 'accuracy']].round(4)
summary['fit_seconds'] = summary['fit_seconds'].round(1)
print(f"(floor) Dummy most-frequent: macro-F1 {dummy_res['macro_f1']:.4f}\n")
summary

(floor) Dummy most-frequent: macro-F1 0.0545



,model,features,n_features,macro_f1,weighted_f1,accuracy,fit_seconds
0,Random Forest,ext,19,0.8074,0.8217,0.8226,1.1
1,XGBoost,ext,19,0.8048,0.8187,0.8185,2.1
2,Gradient Boosting,ext,19,0.8044,0.8144,0.8144,38.6
3,Gradient Boosting,raw,14,0.8019,0.8159,0.8154,25.7
4,Random Forest,raw,14,0.7982,0.8147,0.8170,1.1
5,XGBoost,raw,14,0.7847,0.7968,0.7963,2.0
6,Logistic Regression,ext,19,0.7842,0.7991,0.7947,0.2
7,Logistic Regression,raw,14,0.7781,0.7923,0.7885,0.2


## 8. Per-class F1: which diseases are hard?

**What this does:** one row per configuration, one column per disease, showing that disease's F1 score.

**Why:** the overall macro-F1 hides where the errors are. Earlier runs showed Ankylosing Spondylitis as the weakest class (F1 ≈ 0.6–0.67) and Reactive Arthritis as the smallest. If the same pattern shows up here for the new Gradient Boosting model, that confirms these two are the real methodological targets for the rest of the project (imbalance handling, feature selection, tuning).

**Note:** Reactive Arthritis has only ~83 validation rows, so its F1 moves a lot with a handful of predictions — do not over-read differences of a few hundredths.

In [9]:
pc = pd.DataFrame(per_class, index=disease_names).T.round(3)
pc['min_class_F1'] = pc.min(axis=1)
pc.loc[summary.apply(lambda r: f"{r['model']} ({r['features']})", axis=1)]

,Ankylosing Spondylitis,Normal,Psoriatic Arthritis,Reactive Arthritis,Rheumatoid Arthritis,Sjögren's Syndrome,Systemic Lupus Erythematosus,min_class_F1
Random Forest (ext),0.639,0.854,0.851,0.594,0.843,0.890,0.981,0.594
XGBoost (ext),0.650,0.846,0.854,0.597,0.837,0.874,0.977,0.597
Gradient Boosting (ext),0.627,0.851,0.835,0.622,0.831,0.885,0.981,0.622
Gradient Boosting (raw),0.630,0.851,0.850,0.583,0.833,0.881,0.984,0.583
Random Forest (raw),0.615,0.846,0.843,0.569,0.848,0.886,0.981,0.569
XGBoost (raw),0.597,0.830,0.831,0.571,0.811,0.871,0.981,0.571
Logistic Regression (ext),0.613,0.840,0.823,0.560,0.819,0.869,0.965,0.560
Logistic Regression (raw),0.585,0.840,0.819,0.559,0.818,0.866,0.959,0.559


## 9. Where do the errors go? (best configuration)

**What this does:** takes the configuration with the highest macro-F1 and prints (a) its full classification report and (b) the five most frequent *specific* confusions — "true disease X was predicted as disease Y" — with how large a share of X they represent.

**Why:** this tells us whether errors are spread randomly or concentrated in a few overlapping disease pairs (e.g. Ankylosing Spondylitis vs Psoriatic/Reactive Arthritis), which is the clinically meaningful finding for the paper.

In [10]:
best = summary.iloc[0]
best_key = f"{best['model']} ({best['features']})"
print("Best configuration on this validation split:", best_key, "\n")
print(classification_report(y_val, preds[best_key], target_names=disease_names, digits=3))

cm = confusion_matrix(y_val, preds[best_key], labels=list(range(7)))
off = [(cm[i, j], i, j) for i in range(7) for j in range(7) if i != j]
off.sort(reverse=True)
print("Top 5 confusions (true -> predicted):")
for cnt, i, j in off[:5]:
    print(f"  {disease_names[i]:<30} -> {disease_names[j]:<30} {cnt:>4}  ({cnt / cm[i].sum():.1%} of true {disease_names[i]})")

Best configuration on this validation split: Random Forest (ext) 

                              precision    recall  f1-score   support

      Ankylosing Spondylitis      0.696     0.591     0.639       340
                      Normal      0.877     0.833     0.854       257
         Psoriatic Arthritis      0.832     0.870     0.851       285
          Reactive Arthritis      0.523     0.687     0.594        83
        Rheumatoid Arthritis      0.828     0.857     0.843       456
          Sjögren's Syndrome      0.866     0.916     0.890       296
Systemic Lupus Erythematosus      1.000     0.963     0.981       217

                    accuracy                          0.823      1934
                   macro avg      0.803     0.817     0.807      1934
                weighted avg      0.824     0.823     0.822      1934

Top 5 confusions (true -> predicted):
  Ankylosing Spondylitis         -> Rheumatoid Arthritis             66  (19.4% of true Ankylosing Spondylitis)
  Ankylosi

## 10. Cross-check: recompute macro-F1 by hand

**What this does:** recomputes the best model's macro-F1 directly from the confusion matrix (per-class F1 = 2·TP / (2·TP + FP + FN), then the plain average) and asserts it equals scikit-learn's number.

**Why:** an independent check that the metric we optimise for the rest of the project is exactly what we think it is, and that predictions and labels are aligned.

In [11]:
tp = np.diag(cm).astype(float)
fp = cm.sum(axis=0) - tp
fn = cm.sum(axis=1) - tp
f1_manual = np.mean(2 * tp / (2 * tp + fp + fn))
print(f"macro-F1 by hand: {f1_manual:.6f} | sklearn: {best['macro_f1']:.6f}")
assert abs(f1_manual - best['macro_f1']) < 1e-4
assert cm.sum() == len(y_val)
print("Cross-check passed.")

macro-F1 by hand: 0.807400 | sklearn: 0.807400
Cross-check passed.


## 11. Save the results

**What this does:** writes the summary to `../results/stage1_val_summary.csv`.

In [12]:
out = summary.copy()
out['eval_set'] = 'validation split of training data (n=%d)' % len(y_val)
out.to_csv('../results/stage1_val_summary.csv', index=False)
print("Saved ../results/stage1_val_summary.csv")

Saved ../results/stage1_val_summary.csv


## Summary — what this notebook establishes

* A first, single-split ranking of Logistic Regression, Random Forest, XGBoost and Gradient Boosting on the raw and extended feature sets, **without touching the test set**.
* Which diseases are hardest and which class pairs are confused.
* **Not established here:** which model is best. That requires the fold-safe 5-fold CV of notebook 04, extended with calibration (Brier, ECE) and McNemar's test.